# Credit Card Approval Prediction - Model Training & Comparison

Welcome to the model training notebook. Here, we will:
1. Load and clean the credit card application and credit history datasets.
2. Prepare features (X) and label target (y).
3. Split the data into train and test sets.
4. Train, evaluate, and save **four different models**:
   - Logistic Regression
   - Decision Tree Classifier
   - Random Forest Classifier
   - XGBoost Classifier
5. Save each trained model to the `models/` directory as a `.joblib` file.
6. Compare performance metrics (Accuracy, F1-Score, ROC-AUC) to select the best model.

## Step 1: Import Libraries

In [2]:
pip install xgboost

   ---------------------------------------- 0.0/69.5 MB ? eta -:--:--
   ---------------------------------------- 0.0/69.5 MB ? eta -:--:--
   ---------------------------------------- 0.0/69.5 MB ? eta -:--:--
   ---------------------------------------- 0.5/69.5 MB 2.4 MB/s eta 0:00:29
    --------------------------------------- 1.0/69.5 MB 2.5 MB/s eta 0:00:28
    --------------------------------------- 1.3/69.5 MB 1.9 MB/s eta 0:00:37
    --------------------------------------- 1.6/69.5 MB 1.8 MB/s eta 0:00:39
   - -------------------------------------- 1.8/69.5 MB 1.8 MB/s eta 0:00:38
   - -------------------------------------- 1.8/69.5 MB 1.8 MB/s eta 0:00:38
   - -------------------------------------- 2.1/69.5 MB 1.4 MB/s eta 0:00:47
   - -------------------------------------- 2.6/69.5 MB 1.5 MB/s eta 0:00:46
   - -------------------------------------- 2.9/69.5 MB 1.4 MB/s eta 0:00:47
   -- ------------------------------------- 3.7/69.5 MB 1.7 MB/s eta 0:00:40
   -- --------------


[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
import os
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Machine Learning models
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

# Evaluation metrics
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix,
    roc_auc_score
)

print("All libraries imported successfully!")

All libraries imported successfully!


## Step 2: Load the Datasets

In [4]:
# Set paths relative to workspace root
REPO_ROOT = Path("..").resolve()
DATA_DIR = REPO_ROOT / "data"
MODELS_DIR = REPO_ROOT / "models"

os.makedirs(MODELS_DIR, exist_ok=True)

print("Loading datasets from:", DATA_DIR)
application_df = pd.read_csv(DATA_DIR / "application_record.csv")
credit_df = pd.read_csv(DATA_DIR / "credit_record.csv")

print(f"Application record dimensions: {application_df.shape}")
print(f"Credit record dimensions: {credit_df.shape}")

Loading datasets from: C:\Users\alsye\Desktop\credit-card-approval-prediction\data
Application record dimensions: (438557, 18)
Credit record dimensions: (1048575, 3)


## Step 3: Data Preprocessing & Feature Engineering

Here, we clean the raw tables and merge them by `ID` to create a single cohesive dataset.

In [5]:
# 1. Clean Application Record
app_clean = application_df.copy()
app_clean['OCCUPATION_TYPE'] = app_clean['OCCUPATION_TYPE'].fillna('Unknown')

# Convert DAYS_EMPLOYED to YEARS_EMPLOYED (DAYS_EMPLOYED is negative; 365243 means unemployed)
app_clean['EMPLOYED'] = app_clean['DAYS_EMPLOYED'] != 365243
app_clean['YEARS_EMPLOYED'] = app_clean['DAYS_EMPLOYED'].apply(
    lambda x: max(-x / 365, 0) if x != 365243 else 0
)

# Create additional financial ratios
app_clean['INCOME_EMPLOY_RATIO'] = app_clean.apply(
    lambda row: row['AMT_INCOME_TOTAL'] / row['YEARS_EMPLOYED'] if row['YEARS_EMPLOYED'] > 0 else row['AMT_INCOME_TOTAL'],
    axis=1
)

# Convert DAYS_BIRTH to AGE in years
app_clean['AGE'] = (-app_clean['DAYS_BIRTH'] / 365).astype(int)

# Drop temporary raw days columns
app_clean.drop(columns=['DAYS_EMPLOYED', 'DAYS_BIRTH'], inplace=True)

# 2. Aggregate Credit Record
credit_clean = credit_df.copy()
status_map = {'X': -1, 'C': 0, '0': 0, '1': 1, '2': 2, '3': 3, '4': 4, '5': 5}
credit_clean['STATUS_NUM'] = credit_clean['STATUS'].map(status_map).fillna(0)

# Group monthly records by ID
credit_clean = credit_clean.sort_values(['ID', 'MONTHS_BALANCE'], ascending=[True, False])

def get_trend(x):
    # Average change in credit status over months
    return x.diff().mean()

agg_funcs = {'STATUS_NUM': ['max', 'min', 'mean', 'last', get_trend]}
features_credit = credit_clean.groupby('ID').agg(agg_funcs)
features_credit.columns = ['STATUS_MAX', 'STATUS_MIN', 'STATUS_MEAN', 'STATUS_LAST', 'STATUS_TREND']
features_credit.reset_index(inplace=True)

# Fill trend missing values (happens when there is only one month of balance)
features_credit['STATUS_TREND'] = features_credit['STATUS_TREND'].fillna(0)

# Calculate total number of late months (status > 0)
features_credit['NUM_LATE_MONTHS'] = credit_clean.groupby('ID')['STATUS_NUM'].apply(
    lambda x: (x > 0).sum()
).values

# 3. Merge Datasets
df_merged = app_clean.merge(features_credit, on='ID', how='left')

# Fill missing credit flags for users with no credit records
credit_cols = ['STATUS_MAX', 'STATUS_MIN', 'STATUS_MEAN', 'STATUS_LAST', 'STATUS_TREND', 'NUM_LATE_MONTHS']
df_merged[credit_cols] = df_merged[credit_cols].fillna(0)

print("Datasets successfully processed and merged!")

Datasets successfully processed and merged!


## Step 4: Create the Target Label (y)

We use a multi-rule heuristic to label applicants as either Approved (1) or Denied (0).

In [6]:
def make_target(row):
    income_ok = row['AMT_INCOME_TOTAL'] >= 120000
    employment_ok = row['YEARS_EMPLOYED'] >= 2
    # Clean credit check
    credit_ok = (
        row['STATUS_MAX'] <= 0 and
        row['STATUS_MIN'] <= 0 and
        row['STATUS_MEAN'] <= 0 and
        row['STATUS_LAST'] <= 0 and
        row['NUM_LATE_MONTHS'] <= 1
    )
    family_ok = row['CNT_CHILDREN'] <= 2 and row['CNT_FAM_MEMBERS'] <= 4
    
    # Score based on how many conditions are met
    score = sum([income_ok, employment_ok, credit_ok, family_ok])
    return 1 if score >= 3 else 0

df_merged['TARGET'] = df_merged.apply(make_target, axis=1)

print("Label Distribution (Target):")
print(df_merged['TARGET'].value_counts())
print(df_merged['TARGET'].value_counts(normalize=True) * 100)

Label Distribution (Target):
TARGET
1    388867
0     49690
Name: count, dtype: int64
TARGET
1    88.66966
0    11.33034
Name: proportion, dtype: float64


## Step 5: Encode Categorical Features & Create Train-Test Split

In [7]:
# Drop ID and the Target from the features matrix (X)
X = df_merged.drop(columns=['ID', 'TARGET'])
y = df_merged['TARGET']

# Perform One-Hot Encoding on nominal/categorical columns
categorical_cols = X.select_dtypes(include=['object', 'category', 'bool']).columns.tolist()
X_encoded = pd.get_dummies(X, columns=categorical_cols, drop_first=True)
X_encoded = X_encoded.astype(float)

feature_columns = X_encoded.columns.tolist()
print(f"Total features after encoding: {len(feature_columns)}")

# Split into 80% training and 20% testing
X_train, X_test, y_train, y_test = train_test_split(
    X_encoded, y, test_size=0.2, random_state=42, stratify=y
)

print(f"X_train shape: {X_train.shape}, X_test shape: {X_test.shape}")

Total features after encoding: 55
X_train shape: (350845, 55), X_test shape: (87712, 55)


## Step 6: Train and Evaluate Models One-by-One

We will write a reusable function to print a detailed report for each model and save the model file using `joblib.dump`.

In [10]:
# Dictionary to hold performance metrics
comparison_dict = {}

def evaluate_and_save_model(model, model_name, file_name):
    print(f"=== Training {model_name} ===")
    model.fit(X_train, y_train)
    
    # Predict probabilities and classes
    probs = model.predict_proba(X_test)[:, 1]
    preds = (probs >= 0.6).astype(int)  # using 0.6 threshold
    
    # Calculate metrics
    acc = accuracy_score(y_test, preds)
    prec = precision_score(y_test, preds)
    rec = recall_score(y_test, preds)
    f1 = f1_score(y_test, preds)
    auc = roc_auc_score(y_test, probs)
    
    # Save metrics for comparison
    comparison_dict[model_name] = {
        'Accuracy': acc,
        'Precision': prec,
        'Recall': rec,
        'F1-Score': f1,
        'ROC-AUC': auc
    }
    
    print(f"\nEvaluation Metrics:")
    print(f"Accuracy:  {acc:.4f}")
    print(f"Precision: {prec:.4f}")
    print(f"Recall:    {rec:.4f}")
    print(f"F1-Score:  {f1:.4f}")
    print(f"ROC-AUC:   {auc:.4f}")
    
    print("\nConfusion Matrix:")
    print(confusion_matrix(y_test, preds))
    
    # Save model artifact
    save_path = MODELS_DIR / file_name
    joblib.dump({
        'model': model,
        'feature_columns': feature_columns,
        'threshold': 0.6
    }, save_path)
    print(f"\nSaved model successfully to: {save_path}\n")

### 1. Logistic Regression Model

In [11]:
lr_model = LogisticRegression(max_iter=2000, class_weight='balanced', solver='liblinear', random_state=42)
evaluate_and_save_model(lr_model, "Logistic Regression", "logistic_regression.joblib")

=== Training Logistic Regression ===

Evaluation Metrics:
Accuracy:  0.8409
Precision: 0.9862
Recall:    0.8322
F1-Score:  0.9027
ROC-AUC:   0.9490

Confusion Matrix:
[[ 9029   909]
 [13049 64725]]

Saved model successfully to: C:\Users\alsye\Desktop\credit-card-approval-prediction\models\logistic_regression.joblib



### 2. Decision Tree Classifier

In [12]:
dt_model = DecisionTreeClassifier(max_depth=10, class_weight='balanced', random_state=42)
evaluate_and_save_model(dt_model, "Decision Tree", "decision_tree.joblib")

=== Training Decision Tree ===

Evaluation Metrics:
Accuracy:  1.0000
Precision: 1.0000
Recall:    1.0000
F1-Score:  1.0000
ROC-AUC:   1.0000

Confusion Matrix:
[[ 9938     0]
 [    0 77774]]

Saved model successfully to: C:\Users\alsye\Desktop\credit-card-approval-prediction\models\decision_tree.joblib



### 3. Random Forest Classifier

In [13]:
rf_model = RandomForestClassifier(n_estimators=100, max_depth=12, class_weight='balanced', random_state=42, n_jobs=-1)
evaluate_and_save_model(rf_model, "Random Forest", "random_forest.joblib")

=== Training Random Forest ===

Evaluation Metrics:
Accuracy:  1.0000
Precision: 1.0000
Recall:    1.0000
F1-Score:  1.0000
ROC-AUC:   1.0000

Confusion Matrix:
[[ 9938     0]
 [    0 77774]]

Saved model successfully to: C:\Users\alsye\Desktop\credit-card-approval-prediction\models\random_forest.joblib



### 4. XGBoost Classifier

Note: We compute a class balance scale ratio for XGBoost using the ratio of negative examples to positive examples.

In [14]:
neg_count = (y_train == 0).sum()
pos_count = (y_train == 1).sum()
scale_pos_weight = neg_count / pos_count

xgb_model = XGBClassifier(
    n_estimators=150,
    max_depth=6,
    scale_pos_weight=scale_pos_weight,
    random_state=42,
    n_jobs=-1
)
evaluate_and_save_model(xgb_model, "XGBoost", "xgboost.joblib")

=== Training XGBoost ===

Evaluation Metrics:
Accuracy:  1.0000
Precision: 1.0000
Recall:    1.0000
F1-Score:  1.0000
ROC-AUC:   1.0000

Confusion Matrix:
[[ 9938     0]
 [    3 77771]]

Saved model successfully to: C:\Users\alsye\Desktop\credit-card-approval-prediction\models\xgboost.joblib



## Step 7: Compare Model Results

In [15]:
comparison_df = pd.DataFrame(comparison_dict).T
print("=== Model Performance Comparison ===")
print(comparison_df.to_string())

=== Model Performance Comparison ===
                     Accuracy  Precision    Recall  F1-Score   ROC-AUC
Logistic Regression  0.840866    0.98615  0.832219  0.902669  0.948952
Decision Tree        1.000000    1.00000  1.000000  1.000000  1.000000
Random Forest        1.000000    1.00000  1.000000  1.000000  1.000000
XGBoost              0.999966    1.00000  0.999961  0.999981  1.000000
